In [1]:
import requests
import pandas as pd
API_KEY= ''

search_url= "https://www.googleapis.com/youtube/v3/search"

search_params={
    "part":"snippet",
    "q":"川越",
    "videoCategoryId":19,
    "type":"video",
    "key":API_KEY,
    "relevanceLanguage":"ja",
    "regionCode": "JP",
    "maxResults":50

}
page_token=None
videos=[]
for i in range(10):
 if page_token:
        search_params["pageToken"] = page_token
 response=requests.get(search_url,params=search_params)
 data=response.json()

 video_ids=[]

 for video in data["items"]:
  video_id=video["id"]["videoId"]
  video_ids.append(video_id)

 video_url = "https://www.googleapis.com/youtube/v3/videos"

 video_params={
    "part":"snippet,statistics",
    "id":",".join(video_ids),
    "key":API_KEY
}

 response=requests.get(video_url,params=video_params)
 data_videos=response.json()



 for video in data_videos["items"]:
  video_id=video["id"]

  title=video["snippet"]["title"]
  published_at = video["snippet"]["publishedAt"]
  description= video["snippet"]["description"]
  tags1=video["snippet"].get("tags",[])
  tags=",".join(tags1)

  views = int(video["statistics"].get("viewCount",0))
  likes = int(video["statistics"].get("likeCount",0))
  comments = int(video["statistics"].get("commentCount",0))

  videos.append({
        "video_id": video_id,
        "title": title,
        "description":description,
        "tags":tags,
        "published_at": published_at,
        "views": views,
        "likes": likes,
        "comments": comments
    })
 page_token = data.get("nextPageToken")


df = pd.DataFrame(videos)
df = df.drop_duplicates(subset="video_id")
print("\n===== YouTube動画データ =====")
df["like_rate"] = df["likes"] / df["views"] * 100
df["comment_rate"] = df["comments"] / df["views"] * 100
print(df)



===== YouTube動画データ =====
        video_id                                              title  \
0    Tbp5yDmUX2Q        【川越東武ホテル宿泊記】駅直結で超便利！綺麗な客室と絶品朝食ブッフェを大満喫する一人旅   
1    6L0HiZ0GS-g     【4K】2025年11月2日（日）【驚愕】外国人より日本人が殺到！埼玉県川越小江戸が激混み！   
2    w7Dj4NVHyng                        【なぜ人気？】小江戸と呼ばれる川越の観光スポットを散策   
3    iwBOf4hj9qQ                             【川越】埼玉第3の都市、想像以上に都会だった   
4    XYU10ug3gAU  【川越観光_埼玉】小江戸・川越を1日で100%満喫するおすすめ観光プランを紹介！おすすめスポ...   
..           ...                                                ...   
445  jAt7ZhFrq0s                          『川越』妖怪まち歩き見た後、駄菓子屋横丁へ【4K】   
446  nudo4BYehCI                      【川越水上公園】造波プールに主がいた【プールフィッシング】   
447  xVkUUvKQ6Fg  Chiikawa Mogumogu Honpo Kawagoe Store (ちいかわもぐも...   
448  hqAkIlBb7Sk                                埼玉県川越市大正浪漫夢通りライブカメラ   
449  j_ZYFthwqRc                            【川越氷川神社】 縁むすび風鈴 #shorts   

                                           description  \
0    今回は、埼玉県川越市にある「川越東武ホテル」の宿泊記をお届けします。\n最安値をチェック\n...   
1    

In [3]:
keywords = [
    "川越",
    "日帰り",
    "江戸",
    "グルメ",
    "城",
    "神社",
    "寺",
    "サツマイモ",
    "着物",
    "スイーツ",
    "菓子",
    "東京",
    "祭り",
    "食べ歩き",
    "観光"
]

results = []

for keyword in keywords:

    condition = (
        df["title"].str.contains(keyword, na=False) |
        df["description"].str.contains(keyword, na=False) |
        df["tags"].str.contains(keyword, na=False)
    )

    target = df[condition]

    results.append({
        "キーワード": keyword,
        "動画本数": len(target),
        "平均再生回数": target["views"].mean(),
        "全体との差（平均再生回数）":target["views"].mean()-df["views"].mean(),
        "再生数の中央値": target["views"].median(),
        "全体との差（再生数の中央値）":target["views"].median()-df["views"].median(),
        "平均高評価数": target["likes"].mean(),
        "全体との差（平均高評価数）":target["likes"].mean()-df["likes"].mean(),
        "平均高評価率": target["like_rate"].mean(),
        "全体との差（平均高評価率）":target["like_rate"].mean()-df["like_rate"].mean(),
        "平均コメント率": target["comment_rate"].mean(),
        "全体との差（平均コメント率）": target["comment_rate"].mean()-df["comment_rate"].mean()
    })

keyword_df = pd.DataFrame(results)

keyword_df = keyword_df.sort_values(
    "平均再生回数",
    ascending=False
)

keyword_df

,キーワード,動画本数,平均再生回数,全体との差（平均再生回数）,再生数の中央値,全体との差（再生数の中央値）,平均高評価数,全体との差（平均高評価数）,平均高評価率,全体との差（平均高評価率）,平均コメント率,全体との差（平均コメント率）
8,着物,9,230450.555556,203075.376417,35491.0,31993.0,4535.000000,4155.224490,1.628309,0.419764,0.016376,-0.125193
13,食べ歩き,79,53996.215190,26621.036052,6663.0,3165.0,1027.772152,647.996642,1.343421,0.134876,0.123871,-0.017698
11,東京,65,53869.092308,26493.913169,18733.0,15235.0,517.446154,137.670644,1.316048,0.107503,0.166271,0.024702
3,グルメ,100,42341.480000,14966.300862,6598.5,3100.5,728.230000,348.454490,1.228074,0.019529,0.157984,0.016415
5,神社,122,42024.663934,14649.484796,6756.0,3258.0,607.672131,227.896621,1.380360,0.171815,0.155878,0.014309
14,観光,154,37853.396104,10478.216966,5738.5,2240.5,685.220779,305.445269,1.193042,-0.015503,0.131703,-0.009866
0,川越,437,27579.226545,204.047406,3524.0,26.0,382.560641,2.785131,1.198665,-0.009880,0.141856,0.000287
4,城,76,22807.302632,-4567.876507,1410.0,-2088.0,143.289474,-236.486037,1.392887,0.184342,0.086994,-0.054575
10,菓子,61,22542.704918,-4832.474220,8818.0,5320.0,192.803279,-186.972232,1.228752,0.020207,0.139259,-0.002310
1,日帰り,28,21438.785714,-5936.393424,8161.5,4663.5,278.607143,-101.168367,1.137207,-0.071339,0.119676,-0.021893


In [4]:
keyword_df.to_csv("kawagoe_ja.csv", index=False, encoding="utf-8-sig")

In [7]:
result1=[]
result1.append({
        "動画本数": len(df),
        "平均再生回数": df["views"].mean(),
        "再生数の中央値": df["views"].median(),
        "平均高評価数": df["likes"].mean(),
        "平均高評価率": df["like_rate"].mean(),
        "平均コメント率": df["comment_rate"].mean(),
})
df_a=pd.DataFrame(result1)
df_a


,動画本数,平均再生回数,再生数の中央値,平均高評価数,平均高評価率,平均コメント率
0,366,29242.142077,3361.5,430.631148,1.269144,0.130009
